# ValueLens - data volume check

Run this in the **same Fabric workspace as your Lakehouse**, with the Lakehouse
attached as the notebook's default (Explorer -> Add -> existing Lakehouse).

It reads only. It writes nothing and changes nothing.

It answers three questions:

1. how many licensed users the dashboard will actually count, and why any are excluded
2. what date range your audit data really covers
3. whether the Agents 365 table has landed and which columns are populated

Run all cells and send the full output back.


In [ ]:
# CONFIG - only change these if your tables live elsewhere
LICENSED_TABLE = 'copilot_licensed_users'
AUDIT_TABLE    = 'copilot_interactions_curated'
AGENTS_TABLE   = 'agents_365'
SCHEMA         = None      # e.g. 'dbo' for a schema-enabled Lakehouse; None to auto-detect


In [ ]:
# === resolve table names ======================================================
from pyspark.sql import functions as F

def _resolve(name):
    """Find <name> whether the Lakehouse is schema-enabled or not."""
    cands = []
    if SCHEMA:
        cands.append(f'{SCHEMA}.{name}')
    cands += [name, f'dbo.{name}']
    for c in cands:
        try:
            spark.read.table(c).limit(1).count()
            return c
        except Exception:
            continue
    return None

resolved = {k: _resolve(v) for k, v in
            {'licensed': LICENSED_TABLE, 'audit': AUDIT_TABLE, 'agents': AGENTS_TABLE}.items()}
for k, v in resolved.items():
    print(f'{k:10} -> {v or "NOT FOUND"}')


def _pick(df, exact, contains=None, exclude=()):
    """
    Resolve a column by exact name first, then by substring.

    Exact-first matters: a fuzzy 'licen' match hits
    'Exchange_License_Assign_Date' long before 'Has_license'.
    """
    lower = {c.lower(): c for c in df.columns}
    for name in exact:
        if name.lower() in lower:
            return lower[name.lower()]
    if contains:
        for c in df.columns:
            cl = c.lower()
            if any(x in cl for x in contains) and not any(x in cl for x in exclude):
                return c
    return None


LICENCE_NAMES = ['Has license', 'Has_license', 'HasLicense', 'HasCopilot',
                 'Has Copilot', 'Has_Copilot', 'Has Copilot License']
LICENCE_EXCLUDE = ['assign', 'activity', 'exchange', 'onedrive', 'sharepoint',
                   'skype', 'yammer', 'teams', 'date']

UPN_NAMES = ['UPN_Normalized', 'User Principal Name', 'User_Principal_Name',
             'userPrincipalName', 'UserPrincipalName']

AUDIT_USER_NAMES = ['Audit_UserId', 'UserId', 'UserPrincipalName',
                    'User Principal Name', 'UPN_Normalized']

DATE_NAMES = ['CreationDate', 'ActivityDate', 'CreatedDateTime']


In [ ]:
# === 1. LICENSED USERS  (drives "Total Licensed Users") =======================
# This measure is NOT affected by the date slicer or by incremental refresh.
# A low number here means the table itself is short, or the "Has license" values
# are not ones the dashboard recognises.
ACCEPTED = {'YES', 'TRUE', 'Y', '1'}

t = resolved['licensed']
if not t:
    print(f'{LICENSED_TABLE} not found - the licensed users file has not been landed.')
else:
    df = spark.read.table(t)
    print(f'table   : {t}')
    print(f'rows    : {df.count():,}')
    print(f'columns : {df.columns}')

    lic = _pick(df, LICENCE_NAMES, ['licen', 'copilot'], LICENCE_EXCLUDE)
    upn = _pick(df, UPN_NAMES, ['principal', 'upn'])

    if not lic:
        print('\n** no "Has license" column found - every row will be treated as unlicensed')
    else:
        print(f'\ndistinct values in "{lic}":')
        counted = 0
        for r in df.groupBy(lic).count().orderBy(F.desc('count')).limit(25).collect():
            v = r[lic]
            ok = v is not None and str(v).strip().upper() in ACCEPTED
            counted += r['count'] if ok else 0
            print(f'    {str(v)!r:24} {r["count"]:>8}   {"counted" if ok else "NOT counted"}')
        total = df.count()
        print(f'\ncounted as licensed : {counted:,} of {total:,}')
        if counted < total:
            print(f'** {total - counted:,} rows excluded because the value is not one of')
            print(f'   YES / TRUE / Y / 1 (after upper-casing and trimming)')

    if upn:
        blank = df.filter(F.col(upn).isNull() | (F.trim(F.col(upn)) == '')).count()
        distinct = df.filter(F.col(upn).isNotNull() & (F.trim(F.col(upn)) != '')) \
                     .select(F.lower(F.trim(F.col(upn)))).distinct().count()
        print(f'\nblank UPN rows dropped     : {blank:,}')
        print(f'distinct UPNs after dedupe : {distinct:,}')
        # 'Total Licensed Users' counts DISTINCT UPNs *that also qualify* on
        # Has license, so quote the qualifying figure, not the row count.
        if lic:
            qual = (df.filter(F.upper(F.trim(F.col(f'`{lic}`'))).isin(list(ACCEPTED)))
                      .filter(F.col(f'`{upn}`').isNotNull() & (F.trim(F.col(f'`{upn}`')) != ''))
                      .select(F.lower(F.trim(F.col(f'`{upn}`')))).distinct().count())
            print(f'qualifying distinct UPNs   : {qual:,}')
            print(f'** the dashboard should show about {qual:,} total licensed users')
            if qual != distinct:
                print(f'   ({distinct - qual:,} users are in the file but not counted as licensed)')
        else:
            print(f'** the dashboard should show about {distinct:,} total licensed users')


In [ ]:
# === 2. AUDIT LOG COVERAGE  (drives "Active Licensed Users") ==================
# This IS filtered by the RangeStart / RangeEnd parameters and by the report's
# date slicer. If the slicer sits outside the range below, activity reads as zero.
t = resolved['audit']
if not t:
    print(f'{AUDIT_TABLE} not found - the audit processor has not run.')
else:
    df = spark.read.table(t)
    n = df.count()
    print(f'table : {t}')
    print(f'rows  : {n:,}')

    dcol = _pick(df, DATE_NAMES, ['creationdate', 'activitydate'])
    if dcol:
        agg = df.select(F.min(dcol).alias('lo'), F.max(dcol).alias('hi')).collect()[0]
        days = df.select(F.to_date(F.col(dcol))).distinct().count()
        print(f'\n"{dcol}" range : {agg["lo"]}  ->  {agg["hi"]}')
        print(f'distinct days  : {days:,}')
        print('\n** set the dashboard date slicer inside this range, or it will show zero.')
        print('   rows per month:')
        (df.groupBy(F.date_format(F.col(dcol), 'yyyy-MM').alias('month'))
           .count().orderBy('month').show(36, False))

    ucol = _pick(df, AUDIT_USER_NAMES, ['userid', 'principal'])
    if ucol:
        du = df.select(F.lower(F.trim(F.col(ucol)))).distinct().count()
        print(f'distinct users in audit data : {du:,}')
        print('** "Active Licensed Users" can never exceed the overlap between this')
        print('   set and the licensed-users list above.')


In [ ]:
# === 3. AGENTS 365 ============================================================
t = resolved['agents']
if not t:
    print(f'{AGENTS_TABLE} not found - the Agent 365 lander/ingester has not run.')
else:
    df = spark.read.table(t)
    n = df.count()
    print(f'table   : {t}')
    print(f'rows    : {n:,}')
    print(f'columns : {len(df.columns)}')
    if n:
        print('\npopulated columns:')
        exprs = [F.count(F.when(F.col(f'`{c}`').isNotNull() &
                                (F.trim(F.col(f'`{c}`').cast('string')) != ''), 1)).alias(c)
                 for c in df.columns]
        row = df.agg(*exprs).collect()[0].asDict()
        blank = []
        for c in df.columns:
            v = row[c]
            if v:
                print(f'    {c:44} {v:>7,}/{n:,}')
            else:
                blank.append(c)
        if blank:
            print(f'\nempty ({len(blank)}):')
            for c in blank:
                print(f'    {c}')
            print('\n** empty columns are usually a renamed field in the Agent 365 export.')
            print('   The dashboard shows these as blank rather than failing.')


In [ ]:
# === 4. OVERLAP CHECK =========================================================
# "Active Licensed Users" counts users present in BOTH tables. A mismatch in
# formatting (domain, casing, guest UPNs) shows up here as a small overlap.
lt, at = resolved['licensed'], resolved['audit']
if lt and at:
    ldf, adf = spark.read.table(lt), spark.read.table(at)
    upn = _pick(ldf, UPN_NAMES, ['principal', 'upn'])
    ucol = _pick(adf, AUDIT_USER_NAMES, ['userid', 'principal'])
    if upn and ucol:
        L = ldf.select(F.lower(F.trim(F.col(upn))).alias('u')).distinct()
        A = adf.select(F.lower(F.trim(F.col(ucol))).alias('u')).distinct()
        nl, na = L.count(), A.count()
        both = L.join(A, 'u', 'inner').count()
        print(f'licensed UPNs      : {nl:,}')
        print(f'audit users        : {na:,}')
        print(f'present in both    : {both:,}')
        if both == 0:
            print('\n** NO OVERLAP - "Active Licensed Users" will be zero.')
            print('   The licensed-users export and the audit log are using different')
            print('   identity formats (or different tenants/environments entirely).')
            print('   Compare the samples below - they must match to join.')
        elif both < min(nl, na) * 0.5:
            print('\n** low overlap - the two sources may format identity differently.')
            print('   sample licensed:')
            for r in L.limit(3).collect():
                print(f'      {r["u"]}')
            print('   sample audit:')
            for r in A.limit(3).collect():
                print(f'      {r["u"]}')
    else:
        print('could not find a UPN column in one of the tables')
else:
    print('need both tables for the overlap check')
